# Battery SEM Degradation Classification — Improved Pipeline

**Key improvements over baseline:**
1. **Feature Engineering**: GLCM texture, LBP, morphological crack geometry, multi-scale LoG, finer 3×3 spatial grid
2. **Cross-Validation**: Per-class metrics (macro F1), confusion matrix per fold, hyperparameter tuning inside CV
3. **Ensemble**: Soft-voting ensemble + optional stacking

In [ ]:
import numpy as np
import pandas as pd
import os
from PIL import Image
from scipy import ndimage
from scipy.ndimage import laplace, gaussian_laplace

# Feature extraction
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern

# Models
from sklearn.ensemble import (
    GradientBoostingClassifier, RandomForestClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# Preprocessing & metrics
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

print('All imports OK')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
TRAIN_DIR       = "/kaggle/input/competitions/cadde-2026-battery-sem-degradation-classification/train"
TEST_DIR        = "/kaggle/input/competitions/cadde-2026-battery-sem-degradation-classification/test"
LABELS_FILE     = "/kaggle/input/competitions/cadde-2026-battery-sem-degradation-classification/train_labels.csv"
SUBMISSION_FILE = "/kaggle/input/competitions/cadde-2026-battery-sem-degradation-classification/sample_submission.csv"

RANDOM_STATE = 42
N_FOLDS      = 5

## Feature Extraction

### Feature groups
| Group | Features | Why it helps |
|---|---|---|
| Intensity stats | mean, std, percentiles | Overall brightness / contrast |
| Dark/bright ratios | very_dark, dark_ratio, mid_ratio, bright_ratio | Cracks = dark voids in SEM |
| Gradient / edges | gradient_x/y, multi-scale LoG | Cracks = sharp intensity transitions |
| **GLCM** *(new)* | contrast, dissimilarity, homogeneity, energy, correlation | Gold standard for SEM surface texture |
| **LBP** *(new)* | 10-bin histogram | Micro-texture, surface roughness |
| Texture | local std 7×7 | Fragmentation → chaotic local variation |
| Entropy | histogram entropy | Complex histograms = more degraded |
| **Morphological** *(new)* | n_components, sizes | Crack count / geometry |
| **3×3 grid** *(new)* | region mean variance/range | Spatial distribution of degradation |

In [ ]:
# ── Individual feature extractors ─────────────────────────────────────────────

def intensity_features(arr):
    """Basic intensity statistics."""
    feats = {}
    feats['mean']  = arr.mean()
    feats['std']   = arr.std()
    feats['min']   = arr.min()
    feats['max']   = arr.max()
    feats['range'] = arr.max() - arr.min()
    feats['skewness'] = float(pd.Series(arr.ravel()).skew())
    feats['kurtosis'] = float(pd.Series(arr.ravel()).kurtosis())
    for p in [5, 10, 25, 50, 75, 90, 95]:
        feats[f'p{p}'] = np.percentile(arr, p)
    return feats


def pixel_ratio_features(arr):
    """Dark/bright pixel ratios — cracks appear dark in SEM images."""
    feats = {}
    feats['very_dark']    = (arr < 50).mean()
    feats['dark_ratio']   = (arr < 80).mean()
    feats['mid_ratio']    = ((arr >= 80) & (arr <= 180)).mean()
    feats['bright_ratio'] = (arr > 180).mean()
    feats['dark_bright_ratio'] = (feats['dark_ratio'] + 1e-6) / (feats['bright_ratio'] + 1e-6)
    return feats


def edge_features(arr):
    """Gradient + multi-scale Laplacian of Gaussian."""
    feats = {}
    # Simple gradients
    grad_x = np.abs(np.diff(arr, axis=1)).mean()
    grad_y = np.abs(np.diff(arr, axis=0)).mean()
    feats['gradient_x']    = grad_x
    feats['gradient_y']    = grad_y
    feats['gradient_mean'] = (grad_x + grad_y) / 2
    feats['gradient_ratio'] = (grad_x + 1e-6) / (grad_y + 1e-6)  # anisotropy

    # Multi-scale Laplacian of Gaussian (captures cracks at different widths)
    for sigma in [1, 2, 4]:
        log = np.abs(gaussian_laplace(arr, sigma=sigma))
        feats[f'log_mean_s{sigma}'] = log.mean()
        feats[f'log_std_s{sigma}']  = log.std()
        feats[f'log_max_s{sigma}']  = log.max()
    return feats


def glcm_features(arr):
    """
    Grey-Level Co-occurrence Matrix features.
    Best-in-class for SEM surface texture discrimination.
    Multiple distances + angles → captures anisotropic crack patterns.
    """
    feats = {}
    arr_uint8 = arr.astype(np.uint8)
    # distances=[1,3]: captures both fine and coarse texture
    # angles=[0, 45, 90, 135]: isotropic coverage
    glcm = graycomatrix(
        arr_uint8,
        distances=[1, 3],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256,
        symmetric=True,
        normed=True
    )
    for prop in ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']:
        vals = graycoprops(glcm, prop).flatten()  # shape: (n_distances * n_angles,)
        feats[f'glcm_{prop}_mean'] = vals.mean()
        feats[f'glcm_{prop}_std']  = vals.std()
        feats[f'glcm_{prop}_max']  = vals.max()
    return feats


def lbp_features(arr):
    """
    Local Binary Pattern histogram.
    Captures micro-texture and surface roughness.
    P=8, R=1 is standard; uniform method reduces noise.
    """
    arr_uint8 = arr.astype(np.uint8)
    lbp = local_binary_pattern(arr_uint8, P=8, R=1, method='uniform')
    # uniform LBP has P+2=10 bins
    hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10), density=True)
    return {f'lbp_{i}': v for i, v in enumerate(hist)}


def texture_features(arr):
    """Local std in 7×7 patches — fragmentation = chaotic local variation."""
    patch = ndimage.generic_filter(arr, np.std, size=7)
    return {
        'local_std_mean': patch.mean(),
        'local_std_std':  patch.std(),
        'local_std_max':  patch.max(),
        'local_std_p90':  np.percentile(patch, 90),
    }


def entropy_features(arr):
    """Histogram entropy — degraded images have more complex histograms."""
    hist, _ = np.histogram(arr, bins=32, range=(0, 255))
    hist = hist / hist.sum()
    hist = hist[hist > 0]
    return {'entropy': -np.sum(hist * np.log2(hist))}


def morphological_features(arr):
    """
    Connected component analysis on dark (crack) regions.
    Severe images have more, smaller fragmented components.
    """
    binary = arr < 80  # dark pixels = cracks / voids
    labeled, n_components = ndimage.label(binary)
    if n_components > 0:
        sizes = ndimage.sum(binary, labeled, range(1, n_components + 1))
        sizes = np.array(sizes)
    else:
        sizes = np.array([0])
    return {
        'n_dark_components':    n_components,
        'mean_component_size':  sizes.mean(),
        'max_component_size':   sizes.max(),
        'std_component_size':   sizes.std(),
        'component_size_ratio': (sizes.max() + 1e-6) / (sizes.mean() + 1e-6),  # 1 = uniform fragments
    }


def spatial_grid_features(arr, grid=3):
    """
    3×3 grid of region means — captures spatial distribution of degradation.
    Replaces the coarser 2×2 quadrant variance.
    """
    h, w = arr.shape
    cells = []
    for i in range(grid):
        for j in range(grid):
            cell = arr[i*h//grid:(i+1)*h//grid, j*w//grid:(j+1)*w//grid]
            cells.append(cell.mean())
    cells = np.array(cells)
    return {
        'grid_mean_var':   np.var(cells),
        'grid_mean_range': np.ptp(cells),
        'grid_mean_std':   np.std(cells),
        'grid_min_cell':   cells.min(),
        'grid_max_cell':   cells.max(),
    }


# ── Master extractor ──────────────────────────────────────────────────────────

def extract_features(img_path):
    """Extract all features from a single SEM image."""
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=float)

    feats = {}
    feats.update(intensity_features(arr))
    feats.update(pixel_ratio_features(arr))
    feats.update(edge_features(arr))
    feats.update(glcm_features(arr))        # ← new
    feats.update(lbp_features(arr))         # ← new
    feats.update(texture_features(arr))
    feats.update(entropy_features(arr))
    feats.update(morphological_features(arr))  # ← new
    feats.update(spatial_grid_features(arr))   # ← new (replaces quad_variance)
    return feats


print('Feature extractors defined.')

In [ ]:
# ── Load and extract training features ───────────────────────────────────────
print('Extracting features from training images...')
labels_df = pd.read_csv(LABELS_FILE)

rows = []
for _, row in labels_df.iterrows():
    path  = os.path.join(TRAIN_DIR, row['filename'])
    feats = extract_features(path)
    feats['filename'] = row['filename']
    feats['label']    = row['label']
    feats['fold']     = row['fold']
    rows.append(feats)

train_df     = pd.DataFrame(rows)
feature_cols = [c for c in train_df.columns if c not in ['filename', 'label', 'fold']]

X  = train_df[feature_cols].values
le = LabelEncoder()
y  = le.fit_transform(train_df['label'])

print(f'Classes : {le.classes_}')
print(f'Features: {len(feature_cols)}')
print(f'Samples : {len(X)}')
print(f'\nFeature groups:')
print(f'  Intensity stats  : {len([c for c in feature_cols if c in ["mean","std","min","max","range","skewness","kurtosis"] or c.startswith("p")])} features')
print(f'  Pixel ratios     : {len([c for c in feature_cols if "ratio" in c or "dark" in c or "bright" in c or "mid" in c])} features')
print(f'  Edge / gradient  : {len([c for c in feature_cols if "grad" in c or "log_" in c])} features')
print(f'  GLCM (new)       : {len([c for c in feature_cols if c.startswith("glcm")])} features')
print(f'  LBP  (new)       : {len([c for c in feature_cols if c.startswith("lbp")])} features')
print(f'  Texture          : {len([c for c in feature_cols if "local_std" in c])} features')
print(f'  Entropy          : {len([c for c in feature_cols if c == "entropy"])} features')
print(f'  Morphological (new): {len([c for c in feature_cols if "component" in c or "n_dark" in c])} features')
print(f'  Spatial grid (new) : {len([c for c in feature_cols if c.startswith("grid")])} features')

## Cross-Validation

Improvements over baseline:
- **Macro F1** tracked alongside accuracy (more robust for 3-class, small dataset)
- **Per-fold confusion matrix** to see which classes are confused
- **Hyperparameter search inside CV** (tuned on training fold only — no data leakage)

In [ ]:
# ── Cross-validation with per-fold tuning ─────────────────────────────────────

def run_cv(model_name, model, X, y, train_df, le, n_folds=5, verbose=True):
    """
    5-fold CV using provided folds.
    Returns dict of scores. Prints per-fold metrics if verbose=True.
    """
    fold_accs, fold_f1s = [], []
    all_true, all_preds = [], []

    for fold in range(n_folds):
        val_mask = train_df['fold'] == fold
        X_tr, y_tr   = X[~val_mask], y[~val_mask]
        X_val, y_val = X[val_mask],  y[val_mask]

        scaler  = StandardScaler()
        X_tr_s  = scaler.fit_transform(X_tr)
        X_val_s = scaler.transform(X_val)  # transform only — no fit on val!

        model.fit(X_tr_s, y_tr)
        preds = model.predict(X_val_s)

        acc = accuracy_score(y_val, preds)
        f1  = f1_score(y_val, preds, average='macro')
        fold_accs.append(acc)
        fold_f1s.append(f1)
        all_true.extend(y_val)
        all_preds.extend(preds)

    if verbose:
        print(f'\n{model_name}')
        print(f'  Acc    : {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}')
        print(f'  Macro F1: {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}')
        print('  Aggregated confusion matrix (all folds):')
        cm = confusion_matrix(all_true, all_preds)
        cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
        print(cm_df.to_string())
        print('  Per-class report:')
        print(classification_report(all_true, all_preds, target_names=le.classes_))

    return {
        'acc_mean': np.mean(fold_accs),
        'acc_std':  np.std(fold_accs),
        'f1_mean':  np.mean(fold_f1s),
        'f1_std':   np.std(fold_f1s),
    }


print('CV helper defined.')

In [ ]:
# ── CV: Individual models ──────────────────────────────────────────────────────
print('=' * 60)
print('Individual model cross-validation')
print('=' * 60)

individual_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=0.1),
    'Random Forest':       RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=300, random_state=RANDOM_STATE),
    'SVM (RBF)':           SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
}

cv_results = {}
for name, model in individual_models.items():
    cv_results[name] = run_cv(name, model, X, y, train_df, le)

print('\n' + '=' * 60)
print(f'{"Model":<25} {"Acc":>8} {"±":>6} {"F1":>8} {"±":>6}')
print('-' * 60)
for name, res in cv_results.items():
    print(f'{name:<25} {res["acc_mean"]:>8.4f} {res["acc_std"]:>6.4f} {res["f1_mean"]:>8.4f} {res["f1_std"]:>6.4f}')

In [ ]:
# ── CV: Hyperparameter-tuned Random Forest ────────────────────────────────────
# GridSearchCV is applied INSIDE each training fold — no data leakage.
print('=' * 60)
print('Hyperparameter-tuned Random Forest (inner 3-fold CV)')
print('=' * 60)

rf_param_grid = {
    'n_estimators':   [200, 500],
    'max_depth':      [None, 15, 25],
    'min_samples_leaf': [1, 2],
    'max_features':   ['sqrt', 0.5],
}

fold_accs, fold_f1s = [], []
all_true, all_preds = [], []
best_params_per_fold = []

for fold in range(N_FOLDS):
    val_mask = train_df['fold'] == fold
    X_tr, y_tr   = X[~val_mask], y[~val_mask]
    X_val, y_val = X[val_mask],  y[val_mask]

    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)

    # Inner CV on training fold only
    inner_cv = GridSearchCV(
        RandomForestClassifier(random_state=RANDOM_STATE),
        rf_param_grid,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1
    )
    inner_cv.fit(X_tr_s, y_tr)
    best_params_per_fold.append(inner_cv.best_params_)

    preds = inner_cv.best_estimator_.predict(X_val_s)
    fold_accs.append(accuracy_score(y_val, preds))
    fold_f1s.append(f1_score(y_val, preds, average='macro'))
    all_true.extend(y_val)
    all_preds.extend(preds)

print(f'Tuned RF — Acc: {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}  |  '
      f'Macro F1: {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}')
print('\nBest params per fold:')
for i, p in enumerate(best_params_per_fold):
    print(f'  Fold {i}: {p}')
print('\nAggregated confusion matrix:')
cm_df = pd.DataFrame(confusion_matrix(all_true, all_preds), index=le.classes_, columns=le.classes_)
print(cm_df)

cv_results['Tuned RF'] = {'acc_mean': np.mean(fold_accs), 'f1_mean': np.mean(fold_f1s)}

## Ensemble

Two strategies:
- **Soft Voting** — averages class probabilities from all models. Safe and robust for small datasets (60 samples).
- **Stacking** — uses model predictions as inputs to a meta-learner (LogReg). Higher ceiling but more overfit risk.

Evaluate both in CV, then pick the winner for final submission.

In [ ]:
# ── CV: Soft Voting Ensemble ──────────────────────────────────────────────────
print('=' * 60)
print('Ensemble: Soft Voting')
print('=' * 60)

soft_voter = VotingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE)),
        ('gb',  GradientBoostingClassifier(n_estimators=300, random_state=RANDOM_STATE)),
        ('svm', SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)),
        ('lr',  LogisticRegression(max_iter=1000, C=0.1)),
    ],
    voting='soft'
)

cv_results['Soft Voting Ensemble'] = run_cv(
    'Soft Voting Ensemble', soft_voter, X, y, train_df, le
)

In [ ]:
# ── CV: Stacking Ensemble ─────────────────────────────────────────────────────
print('=' * 60)
print('Ensemble: Stacking (RF + GB + SVM → LogReg meta-learner)')
print('=' * 60)

stacker = StackingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)),
        ('gb',  GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE)),
        ('svm', SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)),
    ],
    final_estimator=LogisticRegression(C=1.0, max_iter=1000),
    cv=3,
    passthrough=True  # passes original features to meta-learner as well
)

cv_results['Stacking Ensemble'] = run_cv(
    'Stacking Ensemble', stacker, X, y, train_df, le
)

In [ ]:
# ── Summary leaderboard ───────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('RESULTS LEADERBOARD (sorted by Macro F1)')
print('=' * 65)
print(f'{"Model":<30} {"Acc":>8} {"Macro F1":>10}')
print('-' * 65)
for name, res in sorted(cv_results.items(), key=lambda x: x[1]['f1_mean'], reverse=True):
    print(f'{name:<30} {res["acc_mean"]:>8.4f} {res["f1_mean"]:>10.4f}')

# Pick best model by macro F1
best_name = max(cv_results, key=lambda x: cv_results[x]['f1_mean'])
print(f'\n→ Best model: {best_name}  (F1: {cv_results[best_name]["f1_mean"]:.4f})')

In [ ]:
# ── Train final model on ALL training data ────────────────────────────────────
print(f'\nTraining final [{best_name}] on all {len(X)} training samples...')

final_scaler = StandardScaler()
X_scaled_all = final_scaler.fit_transform(X)

# Build the final model (same architecture as best)
final_model_map = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=0.1),
    'Random Forest':       RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=300, random_state=RANDOM_STATE),
    'SVM (RBF)':           SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    'Soft Voting Ensemble': VotingClassifier(
        estimators=[
            ('rf',  RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE)),
            ('gb',  GradientBoostingClassifier(n_estimators=300, random_state=RANDOM_STATE)),
            ('svm', SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)),
            ('lr',  LogisticRegression(max_iter=1000, C=0.1)),
        ],
        voting='soft'
    ),
    'Stacking Ensemble': StackingClassifier(
        estimators=[
            ('rf',  RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)),
            ('gb',  GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE)),
            ('svm', SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)),
        ],
        final_estimator=LogisticRegression(C=1.0, max_iter=1000),
        cv=3,
        passthrough=True
    ),
    'Tuned RF': RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE),  # uses default; re-tune below if needed
}

final_model = final_model_map[best_name]
final_model.fit(X_scaled_all, y)
print('Done.')

# Feature importances (for tree-based single models)
base = final_model
if hasattr(base, 'estimators_') and hasattr(list(base.estimators_.values())[0] if isinstance(base.estimators_, dict) else base.estimators_[0], 'feature_importances_'):
    pass  # ensemble — skip
elif hasattr(base, 'feature_importances_'):
    imp_df = pd.DataFrame({
        'feature':    feature_cols,
        'importance': base.feature_importances_
    }).sort_values('importance', ascending=False)
    print('\nTop 15 most important features:')
    print(imp_df.head(15).to_string(index=False))

In [ ]:
# ── Predict on test set ───────────────────────────────────────────────────────
print('Extracting features from test images...')
sub_df = pd.read_csv(SUBMISSION_FILE)

test_rows = []
for filename in sub_df['filename']:
    path  = os.path.join(TEST_DIR, filename)
    feats = extract_features(path)
    feats['filename'] = filename
    test_rows.append(feats)

test_df       = pd.DataFrame(test_rows)
X_test        = test_df[feature_cols].values
X_test_scaled = final_scaler.transform(X_test)  # transform only — never fit on test!

preds_encoded = final_model.predict(X_test_scaled)
preds_labels  = le.inverse_transform(preds_encoded)

print(f'Predictions: {dict(zip(*np.unique(preds_labels, return_counts=True)))}')

In [ ]:
# ── Save submission ───────────────────────────────────────────────────────────
submission = pd.DataFrame({
    'filename': sub_df['filename'],
    'pred':     preds_labels
})
submission.to_csv('submission.csv', index=False)
print('submission.csv saved.')
print(submission.head(10))